# W4D4 — Augmentation A/B — Lab

**Week 4 · Day 4 · CNNs & Model Fine-Tuning** · Lab

Four hundred images, five classes, eighty each. That is not enough, and everybody knows a trick for
it: flip the images, rotate them a bit, crop them differently each epoch, and you have more data for
free.

Today you find out what the trick is actually worth, on this dataset, with a number attached.
Same architecture, same seed, same epochs, one variable — the transform pipeline. Then you do the
same thing with an augmentation that is *wrong* for this data, and watch the accuracy fall below
where it started.

Before any of that, one hygiene task that decides whether the rest of the day means anything:
**twelve of the pizzas are near-duplicates of other pizzas**, and if a pair lands on both sides of
the split, your validation set is scoring the model on images it trained on.

<div dir="rtl" align="right">

# الأسبوع ٤ · اليوم ٤ — زيادة البيانات: تجربة أ/ب

**الأسبوع الرابع · اليوم الرابع · الشبكات الالتفافية وضبط النماذج** · معمل

أربعمئة صورة في خمس فئات، ثمانون لكل فئة. وهذا لا يكفي، والجميع يعرف حيلةً له: اقلب الصور، وأدِرها
قليلًا، واقتطعها اقتطاعًا مختلفًا في كل حقبة، فتصير عندك بيانات أكثر مجّانًا.

واليوم تعرف ما تساويه الحيلة فعلًا على هذه البيانات، برقمٍ ملازم. البنية نفسها والبذرة نفسها والحقب
نفسها ومتغيّر واحد: خطّ التحويلات. ثم تفعل الشيء نفسه بزيادةٍ **خاطئة** لهذه البيانات، وتراقب الدقة
تهبط دون نقطة البداية.

وقبل ذلك كله مهمّة نظافةٍ تقرّر هل لبقيّة اليوم معنى: **اثنتا عشرة من صور البيتزا شبه مكرّرة عن صور
بيتزا أخرى**، وإذا وقع زوج على جانبَي التقسيم صارت مجموعة التحقّق تُقيّم النموذج على صور تدرّب عليها.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Detect near-duplicate images with a perceptual hash, and explain why a checksum finds nothing.
- Split a small dataset so that duplicates cannot straddle it, and keep the class balance.
- Run a controlled A/B where exactly one thing differs, and prove the two runs started identical.
- Say why an augmented run's *training* accuracy is lower and why that is the point.
- Judge whether a transform is valid for a given dataset, and demonstrate one that is not.
- Report a measured gap with the variance around it, rather than a single number.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تكشف الصور شبه المكرّرة بتجزيء إدراكي، وتشرح لماذا لا تجد البصمة الرقمية شيئًا.
- أن تقسم مجموعة صغيرة بحيث لا تستطيع المكرّرات أن تتوزّع بين جانبيها، مع بقاء توازن الفئات.
- أن تُجري تجربة أ/ب مضبوطة يختلف فيها شيء واحد بالضبط، وأن تُثبت أن التشغيلتين بدأتا متطابقتين.
- أن تقول لماذا تكون دقة **التدريب** أقلّ في التشغيلة المزيدة، ولماذا هذا هو المقصود.
- أن تحكم هل التحويل صالح لبيانات بعينها، وأن تُبرهن على واحد غير صالح.
- أن تعرض فجوةً مقيسة ومعها تباينها، لا رقمًا واحدًا.

</div>

## About the data

**Dataset:** `small_image_5class` — 400 images, 224×224 RGB, in `ImageFolder` layout. Five classes
in alphabetical order, which is the order `ImageFolder` assigns: **0 bus, 1 cat, 2 dog, 3 pizza,
4 zebra**, eighty images each.

They are square crops around one annotated object, cut from the CC BY 2.0 / CC BY-SA 2.0 subset of
COCO train2017; `ATTRIBUTION.csv` credits every source photograph. One row is one image, the target
is its folder, and the classes are exactly balanced.

It is small **on purpose**. 320 training images cannot support a network trained from scratch, which
is what makes both of this week's remaining ideas visible: augmentation today, and standing on a
pretrained model tomorrow. The same 400 images carry all of week 6, so the numbers you produce here
stay comparable for a fortnight.

**The known problem: class 3 has twelve near-duplicates.** Each is a 97% crop of another pizza in
the same class, very slightly brighter, re-encoded at a lower JPEG quality. No two files are
byte-identical, so `duplicated()` and a checksum both report a clean dataset. A perceptual hash
finds all twelve within 4 bits, while the closest pair of genuinely different pizzas is 14 bits
apart — a gap wide enough that the threshold is not a guess. Task 2.1 is to find them; a plain
stratified split puts **five of the twelve pairs** on opposite sides.

<div dir="rtl" align="right">

## عن البيانات

**مجموعة البيانات:** `small_image_5class` — أربعمئة صورة ٢٢٤×٢٢٤ ملوّنة بترتيب `ImageFolder`. وخمس
فئات بالترتيب الأبجدي وهو ما تُسنده `ImageFolder`: **٠ حافلة، ١ قطّ، ٢ كلب، ٣ بيتزا، ٤ حمار وحشي**،
ثمانون صورة لكلٍّ منها.

وهي اقتطاعات مربّعة حول كائن مُعلَّم واحد، مقصوصة من المجموعة الجزئية المرخَّصة CC BY 2.0 /
CC BY-SA 2.0 من COCO train2017، وينسب `ATTRIBUTION.csv` كل صورة إلى أصلها. والصف الواحد صورة،
والهدف مجلّدها، والفئات متوازنة تمامًا.

وهي صغيرة **عمدًا**. فثلاثمئة وعشرون صورة تدريب لا تكفي شبكةً تُدرَّب من الصفر، وهذا ما يجعل فكرتَي
هذا الأسبوع الباقيتين مرئيّتين: زيادة البيانات اليوم، والوقوف على نموذج مُدرَّب مسبقًا غدًا. والصور
الأربعمئة نفسها تحمل الأسبوع السادس كله، فتبقى أرقامك قابلة للمقارنة أسبوعين.

**والمشكلة المعروفة: الفئة ٣ فيها اثنتا عشرة صورة شبه مكرّرة.** كلٌّ منها اقتطاع بنسبة ٩٧٪ من صورة
بيتزا أخرى في الفئة نفسها، أنصع قليلًا ومُعاد ترميزها بجودة أقلّ. ولا يتطابق ملفّان بايتًا ببايت،
فتُبلّغ `duplicated()` والبصمة الرقمية عن بيانات نظيفة. أما التجزيء الإدراكي فيجد الاثنتي عشرة كلها
ضمن ٤ بتّات، بينما يبعد أقرب زوج من صور بيتزا مختلفة فعلًا ١٤ بتًّا — وهي فجوة تكفي لألّا تكون العتبة
تخمينًا. والمهمة ٢٫١ أن تجدها؛ والتقسيم الطبقي البسيط يضع **خمسة من الأزواج الاثني عشر** على جانبين
متقابلين.

</div>

## Setup

Images are resized to 128×128 here, not their stored 224. It costs about a point of accuracy and
halves the training time, which is what makes four runs fit inside the session. Tomorrow, with a
pretrained backbone that expects 224, the size goes back up.

<div dir="rtl" align="right">

## الإعداد

تُقاس الصور هنا إلى ١٢٨×١٢٨ لا إلى ٢٢٤ المخزَّنة. وهذا يكلّف نحو نقطة من الدقة ويُنصّف زمن التدريب،
وهو ما يجعل أربع تشغيلات تتّسع داخل الجلسة. وغدًا، مع عمود فقري مُدرَّب مسبقًا يتوقّع ٢٢٤، يعود الحجم
إلى الارتفاع.

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
from aiep.data import get_dataset_dir, describe_dataset
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, check_shape, report

ensure("scikit-learn", "matplotlib")
seed_everything(42)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

IMAGE_ROOT = get_dataset_dir("small_image_5class") / "images"
IMAGE_SIZE = 128
EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
SEED = 42

catalogue = datasets.ImageFolder(IMAGE_ROOT)
CLASSES = catalogue.classes
LABELS = np.array([label for _, label in catalogue.samples])
FILES = [path for path, _ in catalogue.samples]

print(describe_dataset("small_image_5class"))
print(f"\n{len(catalogue)} images | classes {CLASSES} | per class {np.bincount(LABELS)}")
print(versions(), "| device:", device())

## Section 1 — Warm-up: the slide's grid, then eight real images  (≈25 min)

Everything here works. Augmentation on a 4×4 grid of integers first, where you can see exactly what
each transform did, and only then on photographs where it is easy to be fooled.

The horizontal flip is the one to check against the slide: column 0 becomes column 3. A rotation by
90° moves every value to a place you can predict. And a 3×3 random crop keeps nine of the sixteen
numbers and throws seven away — which is the transform that can destroy a label, and the reason
task 2.5 exists.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: شبكة الشريحة ثم ثماني صور حقيقية (نحو ٢٥ دقيقة)

كل ما هنا يعمل. زيادة البيانات على شبكة ٤×٤ من الأعداد الصحيحة أولًا، حيث ترى بالضبط ما فعله كل
تحويل، ثم بعدها على صور يسهل أن تنخدع فيها.

والقلب الأفقي هو ما تتحقّق منه مقابل الشريحة: يصير العمود ٠ العمودَ ٣. والإدارة ٩٠ درجة تنقل كل قيمة
إلى موضع تستطيع التنبّؤ به. والاقتطاع العشوائي ٣×٣ يُبقي تسعة من الأعداد الستة عشر ويرمي سبعة — وهو
التحويل الذي يستطيع إتلاف التسمية، وسبب وجود المهمة ٢٫٥.

</div>

In [ ]:
grid = torch.arange(1, 17, dtype=torch.float).view(1, 4, 4)
print("the slide's grid:\n", grid.squeeze().int().numpy())

flipped = transforms.RandomHorizontalFlip(p=1.0)(grid)
print("\nRandomHorizontalFlip(p=1.0) — columns reversed:\n", flipped.squeeze().int().numpy())

rotated = transforms.RandomRotation((90, 90))(grid)
print("\nRandomRotation(90):\n", rotated.squeeze().int().numpy())

torch.manual_seed(SEED)
cropped = transforms.RandomCrop(3)(grid)
print("\nRandomCrop(3) — nine of the sixteen survive, seven are gone:\n",
      cropped.squeeze().int().numpy())

print("\nevery one of those was a new training example the model had never seen, "
      "made from a grid it had.")

In [ ]:
sample_index = int(np.flatnonzero(LABELS == CLASSES.index("pizza"))[0])
sample = Image.open(FILES[sample_index]).convert("RGB")

demo = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2, 0.2),
])

torch.manual_seed(SEED)
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for index, ax in enumerate(axes.ravel()):
    ax.imshow(sample if index == 0 else demo(sample))
    ax.set_title("original" if index == 0 else f"augmented #{index}")
    ax.axis("off")
plt.suptitle("one image, eight training examples — and one of them lost the plate")
plt.tight_layout()
plt.show()

**Look for the bad one.** At least one of those seven crops has cut away most of the plate, or
zoomed so far into the topping that the picture could be almost any food. That image still carries
the label `pizza`, and the model will be punished for not saying so.

That is the cost side of augmentation, and it is why "more augmentation is better" is false. You
are trading examples the model has never seen against labels that have stopped being true.

<div dir="rtl" align="right">

**ابحث عن السيّئة.** فواحدة على الأقل من تلك الاقتطاعات السبعة قطعت معظم الطبق، أو قرّبت في الطبقة
العليا حتى صارت الصورة تصلح لأي طعام. وتلك الصورة ما زالت تحمل تسمية `pizza`، وسيُعاقَب النموذج إن لم
يقلها.

وهذا هو جانب الكلفة في زيادة البيانات، وهو سبب بطلان «كلما زادت الزيادة كان أفضل». فأنت تقايض أمثلةً
لم يرها النموذج قط بتسمياتٍ لم تعد صحيحة.

</div>

## Section 2 — Core: five tasks  (≈60 min)

1. Find the twelve near-duplicates, then split so no pair can straddle it.
2. Run A — the baseline. No augmentation. It memorises the training set.
3. Run B — the same model, same seed, one variable changed.
4. Both curves on one figure, and the gap written into `aug_ab.parquet`.
5. Which transforms are valid here — and a demonstration of one that is not.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: خمس مهام (نحو ٦٠ دقيقة)

١. جِد الصور شبه المكرّرة الاثنتي عشرة، ثم قسّم بحيث لا يستطيع زوج أن يتوزّع بين الجانبين.
٢. التشغيلة أ — الأساس. بلا زيادة. تحفظ مجموعة التدريب.
٣. التشغيلة ب — النموذج نفسه والبذرة نفسها ومتغيّر واحد مُغيَّر.
٤. المنحنيان في شكل واحد، والفجوة مكتوبة في `aug_ab.parquet`.
٥. أي التحويلات صالح هنا — وبرهانٌ على واحد غير صالح.

</div>

### Task 2.1 — find the duplicates, then split around them

Two steps, and the second is the one people skip.

**Find them.** A difference hash: shrink the image to 9×8 greyscale, compare each pixel with its
right-hand neighbour, and keep the 64 booleans. Two images are near-duplicates when their hashes
differ in **at most 8 of 64 bits**. Small crops, brightness changes and re-encoding barely move
that hash; two different pizzas move it a lot. You should find exactly 12 pairs, all in `pizza`.

**Split around them.** Group every image with anything it duplicates — a pair becomes one unit —
and then split the *units*, stratified by class. Do the naive split first and count how many pairs
it breaks: on this dataset with seed 42, it breaks **five**. Each broken pair puts a near-copy of a
validation image into training, and the validation score goes up for a reason that has nothing to
do with the model.

<div dir="rtl" align="right">

### المهمة ٢٫١ — جِد المكرّرات ثم قسّم حولها

خطوتان، والثانية هي التي يتخطّاها الناس.

**جِدها.** تجزيء الفروق: صغّر الصورة إلى ٩×٨ رمادية، وقارن كل بكسل بجاره الأيمن، واحتفظ بالقيم
المنطقية الأربع والستّين. والصورتان شبه مكرّرتين إذا اختلف تجزيئاهما في **ثمانية بتّات من ٦٤ على
الأكثر**. فالاقتطاعات الصغيرة وتغيّر الإضاءة وإعادة الترميز تكاد لا تحرّك ذلك التجزيء، وصورتا بيتزا
مختلفتان تحرّكانه كثيرًا. ويجب أن تجد اثني عشر زوجًا بالضبط، كلها في `pizza`.

**وقسّم حولها.** اجمع كل صورة مع ما تُكرِّره — فالزوج يصير وحدة واحدة — ثم قسّم **الوحدات** تقسيمًا
طبقيًا بالفئة. وأجرِ التقسيم الساذج أولًا وعُدّ كم زوجًا يكسر: على هذه البيانات بالبذرة ٤٢ يكسر
**خمسة**. وكل زوج مكسور يضع نسخةً شبه مطابقة لصورة تحقّق داخل التدريب، فترتفع درجة التحقّق لسبب لا
علاقة له بالنموذج.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) The hash: PIL to greyscale, resize to (9, 8), np.asarray, then compare
#    pixels[:, 1:] with pixels[:, :-1] and flatten the booleans.
# 2) Compare every pair of hashes and keep those differing in 8 bits or fewer. 400
#    images is 79,800 pairs — a numpy broadcast does it instantly, a Python loop
#    takes a minute.
# 3) Grouping: give every image its own group id, then merge the ids of any pair you
#    found. A tiny union-find, or repeated relabelling — both are fine at this size.
# 4) Split the groups with train_test_split, stratifying on each group's class, then
#    expand the chosen groups back to image indices.
# Search: "perceptual difference hash duplicate images numpy"
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
#
# ١) التجزيء: حوّل بـPIL إلى رمادي، وقِس إلى (٩، ٨)، ثم `np.asarray`،
#    ثم قارن
#    `pixels[:, 1:]` بـ`pixels[:, :-1]` وسطّح القيم المنطقية.
# ٢) قارن كل زوج من التجزيئات وأبقِ ما اختلف في ٨ بتّات فأقلّ. وأربعمئة صورة تعني
#    ٧٩٬٨٠٠ زوج — يفعلها بثّ numpy فورًا، وتأخذ حلقة بايثون دقيقة.
# ٣) التجميع: أعطِ كل صورة معرّف مجموعة خاصًّا بها، ثم ادمج معرّفات أي زوج وجدته.
#    وبنية اتّحاد-بحث صغيرة أو إعادة تسمية متكرّرة، وكلاهما يفي بهذا الحجم.
# ٤) قسّم المجموعات بـ`train_test_split` مع التطبيق الطبقي على فئة كل مجموعة، ثم
#    وسّع المجموعات المختارة إلى فهارس الصور.
# ابحث عن: "perceptual difference hash duplicate images numpy"
# https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
# ────────────────────────────────────────────────────────────────────

from sklearn.model_selection import train_test_split
DUPLICATE_BITS = 8
    # TODO: Greyscale, resize to (size+1, size), compare each pixel with its right neighbour.
    # مهمة: رمادي، ثم قياس إلى (size+1, size)، ثم قارن كل بكسل بجاره الأيمن.
# TODO: they land in.
# مهمة: جزّئ كل صورة، وجِد كل الأزواج ضمن `DUPLICATE_BITS`، واعرض في أي الفئات تقع.
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for column, (left, right) in enumerate(DUPLICATE_PAIRS[:3]):
plt.suptitle("three of the twelve planted pairs — different files, same photograph")
plt.tight_layout(); plt.show()
# TODO: and split the groups instead.
# مهمة: وقسّم المجموعات بدلًا من ذلك.

### Task 2.2 — run A, the baseline

A small CNN, trained from scratch, no augmentation at all: resize, to-tensor, normalise. Thirty
epochs, `Adam` at 1e-3, batches of 32. About 40 seconds.

Write the model builder and the training function so that **every knob is an argument and the seed
is set inside the builder**. Run B differs from run A in one argument, and the only way to be sure
of that is to make it structurally impossible for anything else to change.

Watch what it does: training accuracy walks to 1.000 and stays there, validation stops around 0.54.
It has memorised 316 images. That is not a bug in the code — it is what a network with more
parameters than examples does, and it is the thing augmentation is supposed to attack.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — التشغيلة أ، الأساس

شبكة التفافية صغيرة مُدرَّبة من الصفر بلا أي زيادة: قياس، وتحويل إلى مصفوفة، وتوحيد. ثلاثون حقبة،
و`Adam` عند ١e−٣، ودفعات من ٣٢. نحو أربعين ثانية.

واكتب بانيَ النموذج ودالة التدريب بحيث **يكون كل مقبض وسيطًا وتُضبط البذرة داخل الباني**. فالتشغيلة ب
تختلف عن أ بوسيط واحد، والسبيل الوحيد إلى اليقين من ذلك أن تجعل تغيّر أي شيء آخر مستحيلًا بنيويًا.

وراقب ما تفعله: تمشي دقة التدريب إلى ١٫٠٠٠ وتبقى، ويتوقّف التحقّق قرب ٠٫٥٤. لقد حفظت ٣١٦ صورة. وليس
هذا خللًا في الشيفرة، بل ما تفعله شبكةٌ معاملاتها أكثر من أمثلتها، وهو ما يُفترض أن تهاجمه زيادة
البيانات.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) build_model(seed) sets torch.manual_seed(seed) and returns the network, so two
#    calls with the same seed give byte-identical initial weights.
# 2) Two conv-ReLU-pool blocks (32 then 64 channels), adaptive average pool to 7x7,
#    flatten, linear 128, ReLU, linear 5. Monday's shape, wider.
# 3) run(name, train_transform) builds the datasets with that transform for training
#    and the plain one for validation, trains, and returns a per-epoch DataFrame.
# 4) The validation transform must never contain a random augmentation. Ever.
# Search: "torchvision ImageFolder Subset different transform train val"
# https://pytorch.org/vision/stable/transforms.html
#
# ١) تضبط `build_model(seed)` قيمة `torch.manual_seed(seed)` وتُرجع الشبكة، فيُعطي
#    نداءان بالبذرة نفسها أوزانًا ابتدائية متطابقة بايتًا ببايت.
# ٢) كتلتا التفاف-ReLU-تجميع (٣٢ ثم ٦٤ قناة)، وتجميع متوسّط تكيّفي إلى ٧×٧، وتسطيح،
#    وخطّية ١٢٨، وReLU، وخطّية ٥. شكل الاثنين، أعرض.
# ٣) تبني `run(name, train_transform)` المجموعات بذلك التحويل للتدريب وبالعادي
#    للتحقّق، وتُدرّب، وتُرجع `DataFrame` لكل حقبة.
# ٤) ويجب ألّا يحوي تحويل التحقّق زيادةً عشوائية أبدًا. أبدًا.
# ابحث عن: "torchvision ImageFolder Subset different transform train val"
# https://pytorch.org/vision/stable/transforms.html
# ────────────────────────────────────────────────────────────────────

import time
NORMALISE = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    NORMALISE,
])
    # TODO: Seed, then two conv-ReLU-pool blocks, an adaptive pool, and a two-layer head.
    # مهمة: ابذُر، ثم كتلتا التفاف-ReLU-تجميع، وتجميع تكيّفي، ورأس من طبقتين.
    # TODO: train and validation accuracy per epoch.
    # مهمة: في كل حقبة.

### Task 2.3 — run B, one variable changed

The same call, with an augmentation pipeline in place of `NO_AUGMENTATION`:

```python
RandomResizedCrop(128, scale=(0.8, 1.0))   # framing and scale
RandomHorizontalFlip()                     # left-right, which nothing here depends on
RandomRotation(10)                         # a hand-held camera is never level
ColorJitter(0.2, 0.2, 0.2)                 # lighting and white balance
```

Each line corresponds to a way these photographs could legitimately have differed. That is the test
for whether an augmentation is allowed: **could a real photograph of this class have looked like
this?** A bus photographed from three metres to the left is still a bus. A bus upside down is not a
photograph anybody will ever hand your model.

Same seed, same architecture, same epochs, same split, same validation transform. `INITIAL_WEIGHTS`
above is a snapshot of the untrained parameters, and the sanity check compares it against a fresh
build afterwards — proof that the two runs started from the same place rather than an assertion
that they did.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — التشغيلة ب، متغيّر واحد مُغيَّر

النداء نفسه، مع خطّ زيادةٍ مكان `NO_AUGMENTATION`:

```python
RandomResizedCrop(128, scale=(0.8, 1.0))   # التأطير والمقياس
RandomHorizontalFlip()                     # يمين-يسار، ولا يعتمد عليه شيء هنا
RandomRotation(10)                         # الكاميرا المحمولة باليد ليست مستويةً أبدًا
ColorJitter(0.2, 0.2, 0.2)                 # الإضاءة وموازنة البياض
```

ويقابل كل سطر طريقةً كانت هذه الصور تستطيع أن تختلف بها اختلافًا مشروعًا. وهذا هو اختبار جواز الزيادة:
**هل كان يمكن لصورة حقيقية من هذه الفئة أن تبدو هكذا؟** فالحافلة المصوَّرة من ثلاثة أمتار إلى اليسار
حافلة، والحافلة المقلوبة رأسًا على عقب ليست صورةً سيسلّمها أحد لنموذجك يومًا.

البذرة نفسها والبنية نفسها والحقب نفسها والتقسيم نفسه وتحويل التحقّق نفسه. و`INITIAL_WEIGHTS` أعلاه
لقطةٌ للمعاملات غير المُدرَّبة، ويقارنها فحص السلامة ببناءٍ جديد بعدها — برهانًا على أن التشغيلتين
بدأتا من المكان نفسه، لا تأكيدًا بأنهما فعلتا.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) transforms.Compose with the four augmentations, then ToTensor and NORMALISE.
#    Order matters: the geometric ones work on the PIL image, normalisation last.
# 2) RandomResizedCrop already resizes, so it replaces the plain Resize.
# 3) Call run() with the same seed and epochs, changing only the transform.
# 4) Expect the training accuracy to be LOWER than run A's. That is the mechanism
#    working, not a failure — every epoch shows the model different pictures.
# Search: "torchvision transforms RandomResizedCrop ColorJitter compose order"
# https://pytorch.org/vision/stable/transforms.html
#
# ١) `transforms.Compose` بالزيادات الأربع ثم `ToTensor` ثم `NORMALISE`. والترتيب
#    مهمّ: الهندسية تعمل على صورة PIL، والتوحيد أخيرًا.
# ٢) و`RandomResizedCrop` تُعيد القياس أصلًا، فتحلّ محلّ `Resize` العادية.
# ٣) نادِ `run()` بالبذرة نفسها والحقب نفسها، ولا تغيّر إلا التحويل.
# ٤) وتوقّع أن تكون دقة التدريب **أقلّ** من التشغيلة أ. فتلك الآلية تعمل لا تفشل —
#    إذ ترى الشبكة صورًا مختلفة في كل حقبة.
# ابحث عن: "torchvision transforms RandomResizedCrop ColorJitter compose order"
# https://pytorch.org/vision/stable/transforms.html
# ────────────────────────────────────────────────────────────────────

# TODO: Build the augmentation pipeline described in the task, then run it.
# مهمة: ابنِ خطّ الزيادة الموصوف في المهمة ثم شغّله.

### Task 2.4 — the gap, with a figure and a file

Both validation curves on one figure, and both training curves alongside them so the shape of the
thing is visible: run A's training accuracy pinned at 1.000 with its validation flat underneath,
run B's training accuracy lower and its validation above A's.

**The augmented run's training accuracy is lower and this is the result, not a problem.** Run A
scores 1.000 on its training set because it has memorised 316 fixed images. Run B never sees the
same image twice — every epoch it gets a different crop, flip and colour of each one — so a
training accuracy of 1.000 is not available to it, and the number it does reach is a harder,
more honest one.

Then write `aug_ab.parquet`: one row per run, with the augmentation config as text, the final train
and validation accuracy, the epochs and the seed. Tomorrow loads it as the from-scratch baseline
that transfer learning has to beat.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الفجوة، بشكلٍ وملفّ

منحنيا التحقّق في شكل واحد، ومعهما منحنيا التدريب ليظهر شكل الأمر: دقة تدريب أ مثبَّتة عند ١٫٠٠٠
وتحقّقها مسطَّح تحتها، ودقة تدريب ب أقلّ وتحقّقها فوق تحقّق أ.

**ودقة التدريب في التشغيلة المزيدة أقلّ، وهذه هي النتيجة لا مشكلة.** فالتشغيلة أ تبلغ ١٫٠٠٠ على
مجموعة تدريبها لأنها حفظت ٣١٦ صورة ثابتة. أما ب فلا ترى الصورة نفسها مرّتين — ففي كل حقبة اقتطاع
وقلب ولون مختلف لكل صورة — فدقّة تدريب ١٫٠٠٠ غير متاحة لها، والرقم الذي تبلغه أصعب وأصدق.

ثم اكتب `aug_ab.parquet`: صف لكل تشغيلة، مع إعداد الزيادة نصًّا، والدقة النهائية للتدريب والتحقّق،
والحقب والبذرة. ويُحمّله الغد أساسًا للتدريب من الصفر يجب أن يتجاوزه التعلّم بالنقل.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) One figure, two panels or one — but label the four lines so a reader can tell
#    train from validation and A from B without asking you.
# 2) The gap worth recording is between the two FINAL validation accuracies.
# 3) Build the parquet from a list of dicts: run, augmentation, train_accuracy,
#    val_accuracy, epochs, seed.
# 4) Write it to ARTEFACT_DIR — tomorrow's lab loads it by name.
# Search: "pandas DataFrame to_parquet list of dicts"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html
#
# ١) شكل واحد بلوحتين أو بلوحة — لكن سمِّ الخطوط الأربعة ليميّز القارئ التدريب من
#    التحقّق وأ من ب دون أن يسألك.
# ٢) والفجوة الجديرة بالتسجيل بين دقّتَي التحقّق **النهائيتين**.
# ٣) ابنِ ملفّ parquet من قائمة قواميس: التشغيلة، والزيادة، ودقة التدريب، ودقة
#    التحقّق، والحقب، والبذرة.
# ٤) واكتبه في `ARTEFACT_DIR` — فمعمل الغد يُحمّله بالاسم.
# ابحث عن: "pandas DataFrame to_parquet list of dicts"
# https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html
# ────────────────────────────────────────────────────────────────────

# TODO: Plot both runs' train and validation accuracy against epoch on one figure.
# مهمة: ارسم دقة التدريب والتحقّق للتشغيلتين مقابل الحقبة في شكل واحد.

### Task 2.5 — which transforms are valid, and one that is not

Fill in `VALIDITY` for five transforms: is it safe on *this* dataset, and why. The test is the one
from task 2.3 — could a real photograph of this class have looked like that? Note that the answer
is a property of the data, not of the transform: a horizontal flip is fine for a cat and fatal for
a photograph of handwriting, where it turns a `b` into a `d`.

Then prove it. Train run C with a **vertical flip** — every image upside down — and report where it
lands. On the reference run it scores 0.450 against the baseline's 0.537: nearly nine points *below* the
run with no augmentation at all.

That is the sentence worth leaving with. Augmentation is not free and it is not automatically
positive; it is an assumption about what the world can look like, and a wrong assumption costs more
than no assumption.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — أي التحويلات صالح، وواحدٌ ليس كذلك

املأ `VALIDITY` لخمسة تحويلات: هل هو آمن على **هذه** البيانات ولماذا. والاختبار هو اختبار المهمة
٢٫٣ — هل كان يمكن لصورة حقيقية من هذه الفئة أن تبدو هكذا؟ ولاحظ أن الجواب صفةٌ للبيانات لا للتحويل:
فالقلب الأفقي جيّد للقطّ وقاتل لصورة خطٍّ مكتوب، إذ يحوّل `b` إلى `d`.

ثم برهِن. درّب التشغيلة ج بـ**قلب رأسي** — كل صورة مقلوبة — واعرض أين تحطّ. وفي التشغيل المرجعي تبلغ
٠٫٤٥٠ مقابل ٠٫٥٣٧ للأساس: نحو تسع نقاط **دون** التشغيلة التي بلا أي زيادة.

وهذه هي الجملة الجديرة بأن تخرج بها. زيادة البيانات ليست مجّانية وليست إيجابية تلقائيًا؛ بل هي افتراض
عن الشكل الذي يستطيع العالم أن يكون عليه، والافتراض الخاطئ يكلّف أكثر من انعدام الافتراض.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) For each transform ask: does a real photo of a bus/cat/dog/pizza/zebra ever look
#    like that? If yes it is safe; if no you are teaching the model a lie.
# 2) The vertical-flip run is one call to run() with a transform that flips at p=1.0
#    — everything else identical, as always.
# 3) Compare it to the BASELINE, not to the augmented run. The claim being tested is
#    "worse than no augmentation at all".
# 4) Keep the run: it goes into aug_ab.parquet as a third row.
# Search: "which data augmentations are label preserving"
# https://pytorch.org/vision/stable/transforms.html
#
# ١) اسأل عن كل تحويل: هل تبدو صورة حقيقية لحافلة أو قطّ أو كلب أو بيتزا أو حمار
#    وحشي هكذا يومًا؟ فإن نعم فهو آمن، وإن لا فأنت تعلّم النموذج كذبة.
# ٢) وتشغيلة القلب الرأسي نداء واحد لـ`run()` بتحويل يقلب عند p=1.0 — وكل ما عداه
#    مطابق كالعادة.
# ٣) وقارنها بـ**الأساس** لا بالتشغيلة المزيدة. فالادّعاء المُختبَر «أسوأ من انعدام
#    الزيادة أصلًا».
# ٤) واحتفظ بالتشغيلة: فهي تدخل `aug_ab.parquet` صفًّا ثالثًا.
# ابحث عن: "which data augmentations are label preserving"
# https://pytorch.org/vision/stable/transforms.html
# ────────────────────────────────────────────────────────────────────

    # TODO: For each: "safe" or "unsafe" for THIS dataset, and one clause saying why.
    # مهمة: لكلٍّ منها: «آمن» أو «غير آمن» لهذه البيانات، وعبارة واحدة تقول لماذا.
# TODO: Train a third run with a vertical flip forced on, and compare it to the baseline.
# مهمة: درّب تشغيلة ثالثة بقلب رأسي مفروض، وقارنها بالأساس.

## Section 3 — Stretch: how much is too much  (≈30 min)

Five strengths of the same pipeline — none, light, medium, aggressive, extreme — and validation accuracy
plotted against strength. The curve rises and then falls, and the interesting part is where it
turns.

Think about what "too much" means before you look at the plot. Augmentation manufactures training
examples by assuming the world could have produced them. Mild jitter and small crops stay inside
what a camera does. Aggressive settings — a 0.3-scale crop, 45° of rotation, saturation doubled —
manufacture images that no camera pointed at a bus has ever produced, and the model spends its
capacity learning to classify them. **The training distribution has drifted away from reality, and
validation accuracy is what notices.**

Five runs, about four minutes. Write one sentence on where your peak was and what you think moved
it.

One warning before you read too much into the middle of the curve: the validation set is 80
images, so one image is 1.25 points. Differences of a point or two between neighbouring settings
are noise, and the only movements worth a sentence are the big ones at the two ends.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: كم هو «أكثر من اللازم» (نحو ٣٠ دقيقة)

خمس قوى للخطّ نفسه — بلا، وخفيفة، ومتوسّطة، وشديدة، ومفرطة — ودقة التحقّق مرسومة مقابل القوّة. فيصعد المنحنى
ثم يهبط، والجزء المثير أين يستدير.

وفكّر فيما يعنيه «أكثر من اللازم» قبل أن تنظر إلى الرسم. فزيادة البيانات تصنع أمثلة تدريب بافتراض أن
العالم كان يستطيع إنتاجها. والاهتزاز الخفيف والاقتطاعات الصغيرة تبقى داخل ما تفعله الكاميرا. أما
الإعدادات الشديدة — اقتطاع بمقياس ٠٫٣، وإدارة ٤٥ درجة، وتشبّع مضاعف — فتصنع صورًا لم تُنتجها كاميرا
صُوِّبت إلى حافلة قط، فيُنفق النموذج سعته في تعلّم تصنيفها. **فقد انحرف توزيع التدريب عن الواقع، ودقة
التحقّق هي التي تلاحظ.**

خمس تشغيلات، نحو أربع دقائق. واكتب جملة عن موضع ذروتك وما تظنّه حرّكها.

وتحذير قبل أن تقرأ في وسط المنحنى أكثر مما يحتمل: مجموعة التحقّق ثمانون صورة، فالصورة الواحدة
١٫٢٥ نقطة. والفروق بنقطة أو نقطتين بين إعدادين متجاورين ضجيج، ولا يستحقّ جملةً إلا التحرّك
الكبير في الطرفين.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Define four transforms that vary the SAME knobs by degree: crop scale, rotation
#    angle, jitter strength. Changing which transforms are present as well as how
#    strong they are would confound the sweep.
# 2) Loop them through run() and collect the final validation accuracy of each.
# 3) Plot accuracy against strength as a line with four points, and mark the peak.
# 4) The extreme run should be clearly worse than the light one. If it is not, push it
#    further — 0.05 crop scale, 120 degrees — until the curve turns.
# Search: "data augmentation strength too much distribution shift"
# https://pytorch.org/vision/stable/transforms.html
#
# ١) عرّف أربعة تحويلات تُغيّر **المقابض نفسها** بالدرجة: مقياس الاقتطاع، وزاوية
#    الإدارة، وقوّة الاهتزاز. فتغيير أي التحويلات حاضرة مع قوّتها معًا يخلط المسح.
# ٢) مرّرها في `run()` واجمع دقة التحقّق النهائية
#    لكلٍّ منها.
# ٣) ارسم الدقة مقابل القوّة خطًّا بأربع نقاط، وعلّم الذروة.
# ٤) ويجب أن تكون التشغيلة المفرطة أسوأ بوضوح من الخفيفة. فإن لم تكن فادفعها أبعد —
#    مقياس اقتطاع ٠٫٠٥ ومئة وعشرون درجة — حتى يستدير المنحنى.
# ابحث عن: "data augmentation strength too much distribution shift"
# https://pytorch.org/vision/stable/transforms.html
# ────────────────────────────────────────────────────────────────────

    return transforms.Compose([
        transforms.RandomResizedCrop(IMAGE_SIZE, scale=(scale, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(degrees),
        transforms.ColorJitter(jitter, jitter, jitter),
        transforms.ToTensor(),
        NORMALISE,
    ])
# TODO: Sweep four strengths, keep each run's final validation accuracy, and plot the curve.
# مهمة: امسح أربع قوى، واحتفظ بدقة التحقّق النهائية لكل تشغيلة، وارسم المنحنى.
fig, ax = plt.subplots(figsize=(8, 4.5))
        alpha=0.6, label="train")
peak = strength_sweep.loc[strength_sweep.val_accuracy.idxmax()]
ax.axvline(peak.strength, color="grey", linestyle=":",
           label=f"peak at '{peak.strength}'")
ax.set_xlabel("augmentation strength"); ax.set_ylabel("accuracy")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
    # TODO: Where was your peak, and what does "too much" mean in one sentence?
    # مهمة: أين كانت ذروتك، وما معنى «أكثر من اللازم» في جملة واحدة؟

## Save your artefact

`aug_ab.parquet` — one row per run: the run name, the augmentation description, final train and
validation accuracy, epochs and seed.

**Tomorrow loads this file.** Run A of the fine-tuning lab is a `resnet18` trained from scratch with
the augmentation that won here, and the whole point of that lab is the comparison between it and a
pretrained backbone. Your baseline number needs to be on disk for that comparison to mean anything.

<div dir="rtl" align="right">

## احفظ أثرك

`aug_ab.parquet` — صف لكل تشغيلة: اسم التشغيلة، ووصف الزيادة، والدقة النهائية للتدريب والتحقّق،
والحقب والبذرة.

**ويُحمّل الغد هذا الملف.** فالتشغيلة أ في معمل الضبط الدقيق شبكة `resnet18` مُدرَّبة من الصفر بالزيادة
التي فازت هنا، ومقصد ذلك المعمل كلّه المقارنة بينها وبين عمود فقري مُدرَّب مسبقًا. فرقمك الأساس يحتاج
أن يكون على القرص ليكون لتلك المقارنة معنى.

</div>

In [ ]:
runs = pd.concat([baseline, augmented, upside_down], ignore_index=True)

summary = pd.DataFrame([
    {"run": "A_baseline", "augmentation": "none (resize + normalise only)",
     "train_accuracy": float(baseline.train_accuracy.iloc[-1]),
     "val_accuracy": BASELINE_VAL, "epochs": EPOCHS, "seed": SEED, "image_size": IMAGE_SIZE},
    {"run": "B_augmented",
     "augmentation": "crop(0.8-1.0) + hflip + rot(10) + jitter(0.2)",
     "train_accuracy": float(augmented.train_accuracy.iloc[-1]),
     "val_accuracy": AUGMENTED_VAL, "epochs": EPOCHS, "seed": SEED, "image_size": IMAGE_SIZE},
    {"run": "C_vertical_flip", "augmentation": "vertical flip p=1.0 (invalid for this data)",
     "train_accuracy": float(upside_down.train_accuracy.iloc[-1]),
     "val_accuracy": INVALID_VAL, "epochs": EPOCHS, "seed": SEED, "image_size": IMAGE_SIZE},
])

AUG_AB_PATH = ARTEFACT_DIR / "aug_ab.parquet"
summary.to_parquet(AUG_AB_PATH, index=False)
runs.to_parquet(ARTEFACT_DIR / "aug_curves.parquet", index=False)

print(summary.round(3).to_string(index=False))
print(f"\nwrote {AUG_AB_PATH.name} — tomorrow's run A has to beat "
      f"{summary.val_accuracy.max():.3f}")

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(len(DUPLICATE_PAIRS) == 12 and STRADDLES == 0,
      f"all 12 near-duplicate pairs must be found and none may straddle the split — found "
      f"{len(DUPLICATE_PAIRS)} pairs, {STRADDLES} straddling (a naive split straddles "
      f"{NAIVE_STRADDLES})",
      f"يجب إيجاد أزواج شبه التكرار الاثني عشر كلها وألّا يتوزّع أيٌّ منها بين الجانبين — الموجود "
      f"{len(DUPLICATE_PAIRS)} زوجًا و{STRADDLES} متوزّعًا (والتقسيم الساذج يوزّع {NAIVE_STRADDLES})")

fresh_weights = torch.cat([p.detach().flatten() for p in build_model(SEED).parameters()])
check(torch.equal(INITIAL_WEIGHTS, fresh_weights),
      "runs A, B and C must start from identical initial weights — rebuilding the model with the "
      "same seed no longer reproduces them, so the comparison has two variables in it",
      "يجب أن تبدأ التشغيلات أ وب وج من أوزان ابتدائية متطابقة — وإعادة بناء النموذج بالبذرة نفسها "
      "لم تعد تُعيد إنتاجها، فصار في المقارنة متغيّران")

check(AUGMENTED_VAL > BASELINE_VAL,
      f"the augmented run's validation accuracy must beat the baseline's — got "
      f"{AUGMENTED_VAL:.3f} against {BASELINE_VAL:.3f}",
      f"يجب أن تتجاوز دقة التحقّق في التشغيلة المزيدة دقة الأساس — والناتج "
      f"{AUGMENTED_VAL:.3f} مقابل {BASELINE_VAL:.3f}")

check(augmented.train_accuracy.iloc[-1] < baseline.train_accuracy.iloc[-1],
      f"the augmented run's TRAIN accuracy must be lower — it never sees the same image twice — "
      f"got {augmented.train_accuracy.iloc[-1]:.3f} against {baseline.train_accuracy.iloc[-1]:.3f}",
      f"يجب أن تكون دقة **التدريب** في التشغيلة المزيدة أقلّ — فهي لا ترى الصورة نفسها مرّتين — "
      f"والناتج {augmented.train_accuracy.iloc[-1]:.3f} مقابل "
      f"{baseline.train_accuracy.iloc[-1]:.3f}")

check(INVALID_VAL < BASELINE_VAL,
      f"the vertical-flip run must score below the baseline — an invalid augmentation is worse "
      f"than none — got {INVALID_VAL:.3f} against {BASELINE_VAL:.3f}",
      f"يجب أن تسجّل تشغيلة القلب الرأسي دون الأساس — فالزيادة غير الصالحة أسوأ من انعدامها — "
      f"والناتج {INVALID_VAL:.3f} مقابل {BASELINE_VAL:.3f}")

random_in_validation = [t for t in VAL_TRANSFORM.transforms
                        if type(t).__name__.startswith("Random")]
check(not random_in_validation,
      f"the validation pipeline must contain no random augmentation — found "
      f"{[type(t).__name__ for t in random_in_validation]}. This is the most common silent error "
      f"in the lab: it makes every run's score noisy and none of them comparable",
      f"يجب ألّا يحوي خطّ التحقّق أي زيادة عشوائية — والموجود "
      f"{[type(t).__name__ for t in random_in_validation]}. وهذا أشيع خطأ صامت في المعمل: يجعل "
      f"درجة كل تشغيلة مضطربة ولا شيء منها قابلًا للمقارنة")

check(AUG_AB_PATH.exists() and len(summary) == 3
      and set(summary.columns) >= {"run", "augmentation", "train_accuracy", "val_accuracy",
                                   "epochs", "seed"},
      f"aug_ab.parquet needs one row per run with the six recorded columns — got "
      f"{len(summary)} rows and {list(summary.columns)}",
      f"يحتاج `aug_ab.parquet` صفًّا لكل تشغيلة بالأعمدة الستة — والموجود {len(summary)} صفوف و"
      f"{list(summary.columns)}")

check(all(len(v.split()) >= 10 for v in VALIDITY.values()) and len(TOO_MUCH.split()) >= 15,
      f"tasks 2.5 and the stretch want written judgements — got "
      f"{ {k: len(v.split()) for k, v in VALIDITY.items()} } and {len(TOO_MUCH.split())} words",
      f"تريد المهمة ٢٫٥ والقسم الإضافي أحكامًا مكتوبة — والموجود "
      f"{ {k: len(v.split()) for k, v in VALIDITY.items()} } و{len(TOO_MUCH.split())} كلمة")

report()

## What's next

**W4D5 — Fine-tuning ResNet-18.** Today's best number came from training a network from scratch on
316 images. Tomorrow you stop doing that: a `resnet18` that already learned what edges and textures
and eyes look like, from a million photographs, gets a new five-class head — and beats today's
result in its **first epoch**.

Your `aug_ab.parquet` is the number it has to beat, and the run that fails tomorrow — the one with
the wrong learning rate — fails in exactly the way today's vertical flip did: confidently, and
below where you started.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٤ اليوم ٥ — الضبط الدقيق لـResNet-18.** جاء أفضل رقم اليوم من تدريب شبكة من الصفر على ٣١٦
صورة. وغدًا تتوقّف عن ذلك: فشبكة `resnet18` تعلّمت مسبقًا شكل الحواف والقوام والعيون من مليون صورة،
تأخذ رأسًا جديدًا بخمس فئات — فتتجاوز نتيجة اليوم في **حقبتها الأولى**.

و`aug_ab.parquet` عندك هو الرقم الذي عليها تجاوزه، والتشغيلة التي تفشل غدًا — تلك ذات معدّل التعلّم
الخاطئ — تفشل تمامًا كما فشل القلب الرأسي اليوم: بثقة، ودون نقطة البداية.

</div>